In [ ]:
import sys
sys.path.append('C:/project_WWTP/python')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from dataclasses import dataclass
from scipy.stats import zscore

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
BASE_DIR = Path.cwd().resolve().parent.parent

DATA_DIR    = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results" / "DL"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_RATIOS = {
    "train": 0.7,
    "val":   0.2,
    "test":  0.1,
}

TIME_COL = "SYS_TIME"

# ── 분석 대상 지표 (단일 분석 셀에서 사용) ──────────────────────
# 선택 가능: "TOC_VU", "SS_VU", "TN_VU", "TP_VU", "FLUX_VU", "PH_VU"
TARGET = "SS_VU"

# ── 지표별 고농도(고이상) 임계값 및 단위 ─────────────────────────
HIGH_THR_MAP = {
    "TOC_VU":  (10.0, "mg/L"),
    "SS_VU":   ( 5.0, "mg/L"),
    "TN_VU":   ( 5.0, "mg/L"),
    "TP_VU":   ( 0.3, "mg/L"),
    "FLUX_VU": ( 5.0, "m³/min"),
    "PH_VU":   ( 7.5, ""),
}

RAIN_THR = 1.0  # 강수 이벤트 임계값 (mm/30min, 3관측소 합산)
COLORS = {"train": "#2196F3", "val": "#FF9800", "test": "#4CAF50"}

In [ ]:
def load_data(DATA_DIR):
    dfs = {}
    dfs['flow'] = pd.read_csv(DATA_DIR / "actual/FLOW_Actual.csv")
    dfs['flow']['Q_in'] = dfs['flow']["flow_TankA"] + dfs['flow']['flow_TankB']
    dfs['flow']['level_sum'] = dfs['flow']['level_TankA'] + dfs['flow']['level_TankB']
    dfs['flow'] = dfs['flow'].drop(columns=["data_save_dt"])
    dfs['tms'] = pd.read_csv(DATA_DIR / "actual/TMS_Actual.csv")
    for station_id in ["368", "541", "569"]:
        aws_path = DATA_DIR / f"actual/AWS_{station_id}.csv"
        df = pd.read_csv(aws_path)
        if "datetime" in df.columns:
            time_col = df["datetime"]
            df = df.drop(columns=["datetime", "YYMMDDHHMI", "STN"], errors="ignore")
            df = df.add_suffix(f"_{station_id}")
            df["SYS_TIME"] = time_col
        else:
            df = df.drop(columns=["YYMMDDHHMI", "STN"], errors="ignore")
            df = df.add_suffix(f"_{station_id}")
        dfs[f"aws{station_id}"] = df
    return dfs


def set_datetime_index(df, time_col):
    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], errors="coerce")
    out = out.dropna(subset=[time_col])
    out = out.set_index(time_col).sort_index()
    return out


def align_data(dfs):
    aligned_dfs = {}
    for name, df in dfs.items():
        df_aligned = set_datetime_index(df, TIME_COL)
        df_aligned = df_aligned.resample("1min").ffill()
        aligned_dfs[name] = df_aligned
    return aligned_dfs


def merge_data(dfs):
    valid = {}
    merged_dfs = {}
    for name, df in dfs.items():
        df2 = df.sort_index()
        if df2.index.has_duplicates:
            df2 = df2[~df2.index.duplicated(keep="last")]
        valid[name] = df2
    for name, df in valid.items():
        if name in ("flow", "tms"):
            merged_dfs[name] = pd.concat(
                [df, valid["aws368"], valid["aws541"], valid["aws569"]],
                axis=1, join="inner"
            )
    return merged_dfs

In [ ]:
@dataclass
class ImputationConfig:
    short_term_hours: int = 3
    medium_term_hours: int = 12
    long_term_hours: int = 48
    ewma_span: int = 6


@dataclass
class OutlierConfig:
    method: str = "iqr"
    iqr_threshold: float = 1.5
    zscore_threshold: float = 3.0
    require_both: bool = True


def impute_missing(df, freq="1h", config=ImputationConfig()):
    df_out = df.copy()
    freq_td = pd.Timedelta(freq)
    freq_hours = freq_td.total_seconds() / 3600
    mask_dict = {}
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        series = df[col].copy()
        original_missing = series.isna()
        mask_dict[f"{col}_is_missing"] = original_missing.astype(int)
        limit_short = max(1, int(config.short_term_hours / freq_hours))
        series_ffill = series.ffill(limit=limit_short)
        ffill_mask = original_missing & ~series_ffill.isna()
        mask_dict[f"{col}_imputed_ffill"] = ffill_mask.astype(int)
        still_missing = series_ffill.isna()
        if still_missing.sum() > 0:
            ewma_span = max(1, int(config.ewma_span / freq_hours))
            series_ewma = series_ffill.ewm(span=ewma_span, adjust=False).mean()
            limit_medium = max(1, int(config.medium_term_hours / freq_hours))
            missing_groups = (still_missing != still_missing.shift()).cumsum()
            missing_lengths = still_missing.groupby(missing_groups).transform("sum")
            medium_mask = (still_missing & (missing_lengths > limit_short)
                           & (missing_lengths <= limit_medium))
            series_ffill[medium_mask] = series_ewma[medium_mask]
            mask_dict[f"{col}_imputed_ewma"] = medium_mask.astype(int)
        else:
            mask_dict[f"{col}_imputed_ewma"] = pd.Series(0, index=df.index, dtype=int)
        still_missing_long = series_ffill.isna()
        if still_missing_long.sum() > 0:
            long_ewma_span = max(1, int(config.ewma_span * 4 / freq_hours))
            series_long_ewma = series_ffill.ewm(span=long_ewma_span, adjust=False).mean()
            series_ffill[still_missing_long] = series_long_ewma[still_missing_long]
            mask_dict[f"{col}_imputed_long_ewma"] = still_missing_long.astype(int)
        else:
            mask_dict[f"{col}_imputed_long_ewma"] = pd.Series(0, index=df.index, dtype=int)
        df_out[col] = series_ffill
    df_mask = pd.DataFrame(mask_dict, index=df.index)
    return df_out, df_mask


def imputate_data(dfs):
    imputed_dfs, mask_imputed_dfs = {}, {}
    for name, df in dfs.items():
        df_imputed, mask_imputed = impute_missing(df, freq="1min", config=ImputationConfig())
        imputed_dfs[name] = df_imputed
        mask_imputed_dfs[name] = mask_imputed
    return imputed_dfs, mask_imputed_dfs

In [ ]:
def outliers_domain(series, col_name):
    outliers = pd.Series([False] * len(series), index=series.index)
    if not pd.api.types.is_numeric_dtype(series):
        return outliers
    regulatory_rules = {
        "TOC_VU":  (0, 30),
        "SS_VU":   (0, 20),
        "PH_VU":   (4.3, 10.0),
        "TN_VU":   (0, 20),
        "TP_VU":   (0, 1.0),
        "FLUX_VU": (0, 34.2),
    }
    if col_name in regulatory_rules:
        lower, upper = regulatory_rules[col_name]
        return (series < lower) | (series > upper)
    physical_rules = {
        "level_TankA": (0, 10), "level_TankB": (0, 10),
        "TA": (-30, 45), "HM": (0, 100), "TD": (-40, 35),
    }
    if col_name in physical_rules:
        lower, upper = physical_rules[col_name]
        outliers = (series < lower) | (series > upper)
    elif "RN_" in col_name:
        outliers = (series < 0) | (series > 300)
    else:
        valid = series.dropna()
        if len(valid) > 0:
            outliers = (series < 0) | (series > valid.quantile(0.999) * 2)
    return outliers


def outliers_statistical(series, method="iqr", iqr_threshold=1.5, zscore_threshold=3.0):
    outliers = pd.Series([False] * len(series), index=series.index)
    if not pd.api.types.is_numeric_dtype(series):
        return outliers
    if method == "iqr":
        Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
        IQR = Q3 - Q1
        outliers = (series < Q1 - iqr_threshold * IQR) | (series > Q3 + iqr_threshold * IQR)
    elif method == "zscore":
        valid_mask = ~series.isna()
        if valid_mask.sum() > 0:
            z_scores = np.abs(zscore(series[valid_mask]))
            outliers[valid_mask] = z_scores > zscore_threshold
    return outliers


def process_outliers(df, config=OutlierConfig(), ewma_span=12):
    df_out = df.copy()
    mask_dict = {}
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        series = df[col].copy()
        domain_out = outliers_domain(series, col)
        stats_out  = outliers_statistical(series, method=config.method,
                                          iqr_threshold=config.iqr_threshold,
                                          zscore_threshold=config.zscore_threshold)
        final_out = (domain_out & stats_out) if config.require_both else (domain_out | stats_out)
        mask_dict[f"{col}_outlier_final"] = final_out.astype(int)
        if final_out.sum() > 0:
            series_clean = series.copy()
            series_clean[final_out] = np.nan
            series[final_out] = series_clean.ewm(span=ewma_span, adjust=False).mean()[final_out]
        df_out[col] = series
    return df_out, pd.DataFrame(mask_dict, index=df.index)


def handle_outliers(dfs):
    processed_dfs, mask_dfs = {}, {}
    for name, df in dfs.items():
        processed_dfs[name], mask_dfs[name] = process_outliers(df)
    return processed_dfs, mask_dfs


def resample_data(df, freq="30min", rain_cols=None, other_cols=None):
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    agg_dict = {}
    for col in numeric_cols:
        if col.startswith("RN_") or col.startswith("AR_") or col == "FLUX_VU":
            agg_dict[col] = "sum"
        else:
            agg_dict[col] = "mean"
    return df[numeric_cols].resample(freq).agg(agg_dict)

In [ ]:
# ====== TMS 지표 분포 분석: Train / Val / Test 구간 비교 ======

from scipy.stats import gaussian_kde


def load_tms_resampled():
    """30min 리샘플 수준의 TMS 데이터 반환 (feature engineering 생략)"""
    dfs = load_data(DATA_DIR)
    aligned = align_data(dfs)
    # FLUX 차분 처리 (누적 → 증분)
    if "tms" in aligned and "FLUX_VU" in aligned["tms"].columns:
        flux = aligned["tms"]["FLUX_VU"].copy()
        fd = flux.diff()
        fd[fd < 0] = flux[fd < 0]
        fd.iloc[0] = 0
        aligned["tms"]["FLUX_VU"] = fd.clip(lower=0)
    merged = merge_data(aligned)
    imputed, _ = imputate_data(merged)
    processed, _ = handle_outliers(imputed)
    return resample_data(processed["tms"], freq="30min")


def analyze_target(target_col, tms_30):
    """target_col에 대한 Train/Val/Test 분포 분석 및 시각화"""
    if target_col not in tms_30.columns:
        print(f"[SKIP] '{target_col}' 컬럼이 데이터에 없습니다.")
        return

    high_thr, unit = HIGH_THR_MAP.get(target_col, (None, ""))
    ylabel = f"{target_col} ({unit})" if unit else target_col

    series = tms_30[target_col].dropna()
    T = len(series)
    n_tr = int(T * SPLIT_RATIOS["train"])
    n_va = int(T * SPLIT_RATIOS["val"])
    s_train = series.iloc[:n_tr]
    s_val   = series.iloc[n_tr : n_tr + n_va]
    s_test  = series.iloc[n_tr + n_va :]

    rain_cols = [c for c in tms_30.columns if c.startswith("RN_")]
    rain_sum  = tms_30[rain_cols].sum(axis=1).reindex(series.index)

    split_date_val  = s_val.index[0]
    split_date_test = s_test.index[0]

    # ── 통계 요약 출력 ──────────────────────────────────────────
    ratio_str = (f"Train {int(SPLIT_RATIOS['train']*100)}/"
                 f"Val {int(SPLIT_RATIOS['val']*100)}/"
                 f"Test {int(SPLIT_RATIOS['test']*100)}")
    print("=" * 65)
    print(f"{target_col} 구간별 통계 (30min 리샘플, 이상치 처리 후)  [{ratio_str}]")
    print("=" * 65)
    for name, s in [("TRAIN", s_train), ("VAL  ", s_val), ("TEST ", s_test)]:
        rain_s  = rain_sum.reindex(s.index).fillna(0)
        rain_ev = (rain_s > RAIN_THR).sum()
        print(f"\n[{name}]  n={len(s):,}  ({s.index[0].date()} ~ {s.index[-1].date()})")
        print(f"  mean={s.mean():.3f}  std={s.std():.3f}  "
              f"p25={s.quantile(0.25):.3f}  p50={s.median():.3f}  "
              f"p75={s.quantile(0.75):.3f}  max={s.max():.3f}")
        print(f"  강수이벤트(>{RAIN_THR}mm): {rain_ev}건 ({rain_ev/len(s)*100:.1f}%)")
        if high_thr is not None:
            high_ev = (s > high_thr).sum()
            print(f"  고농도(>{high_thr}{unit}): {high_ev}건 ({high_ev/len(s)*100:.1f}%)")

    # ── 시각화 ──────────────────────────────────────────────────
    fig = plt.figure(figsize=(18, 14))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

    # (A) 전체 시계열
    ax_ts = fig.add_subplot(gs[0, :])
    for s, lbl, c in [(s_train, "Train", COLORS["train"]),
                      (s_val,   "Val",   COLORS["val"]),
                      (s_test,  "Test",  COLORS["test"])]:
        ax_ts.plot(s.index, s.values, color=c, lw=0.7, label=lbl)
    ax_ts.axvline(split_date_val,  color="gray",  ls="--", lw=1.5,
                  label=f"Val 시작 ({split_date_val.date()})")
    ax_ts.axvline(split_date_test, color="black", ls="--", lw=1.5,
                  label=f"Test 시작 ({split_date_test.date()})")
    if high_thr is not None:
        ax_ts.axhline(high_thr, color="red", ls=":", lw=1, alpha=0.6,
                      label=f"임계값 ({high_thr}{unit})")
    ax_ts.set_title(f"{target_col} 전체 시계열 (Train / Val / Test)",
                    fontsize=12, fontweight="bold")
    ax_ts.set_ylabel(ylabel)
    ax_ts.legend(loc="upper right", fontsize=8, ncol=3)
    ax_ts.grid(True, alpha=0.3)

    # (B) 분포 히스토그램 + KDE
    ax_hist = fig.add_subplot(gs[1, 0])
    for lbl, s, c in [("Train", s_train, COLORS["train"]),
                      ("Val",   s_val,   COLORS["val"]),
                      ("Test",  s_test,  COLORS["test"])]:
        ax_hist.hist(s, bins=40, density=True, alpha=0.3, color=c)
        xs = np.linspace(s.min(), s.max(), 300)
        ax_hist.plot(xs, gaussian_kde(s.dropna())(xs), color=c, lw=2, label=lbl)
    if high_thr is not None:
        ax_hist.axvline(high_thr, color="red", ls=":", lw=1.5, alpha=0.7)
    ax_hist.set_title("분포 (KDE)", fontsize=11, fontweight="bold")
    ax_hist.set_xlabel(ylabel)
    ax_hist.set_ylabel("밀도")
    ax_hist.legend()
    ax_hist.grid(True, alpha=0.3)

    # (C) 박스플롯
    ax_box = fig.add_subplot(gs[1, 1])
    bp = ax_box.boxplot(
        [s_train.values, s_val.values, s_test.values],
        labels=["Train", "Val", "Test"],
        patch_artist=True, notch=True,
        medianprops=dict(color="red", lw=2),
    )
    for patch, c in zip(bp["boxes"], COLORS.values()):
        patch.set_facecolor(c); patch.set_alpha(0.6)
    if high_thr is not None:
        ax_box.axhline(high_thr, color="red", ls=":", lw=1.5, alpha=0.7,
                       label=f"임계값 ({high_thr})")
    ax_box.set_title("박스플롯", fontsize=11, fontweight="bold")
    ax_box.set_ylabel(ylabel)
    ax_box.legend(fontsize=8)
    ax_box.grid(True, alpha=0.3, axis="y")

    # (D) 월별 평균
    ax_mon = fig.add_subplot(gs[2, 0])
    s_all   = pd.concat([s_train, s_val, s_test])
    monthly = s_all.groupby(s_all.index.month).agg(["mean", "std"])
    ax_mon.bar(monthly.index, monthly["mean"], yerr=monthly["std"],
               capsize=4, color="#9C27B0", alpha=0.7, label="전체 평균±std")
    for lbl, s, c in [("Train", s_train, COLORS["train"]),
                      ("Val",   s_val,   COLORS["val"]),
                      ("Test",  s_test,  COLORS["test"])]:
        m_mean = s.groupby(s.index.month).mean()
        ax_mon.plot(m_mean.index, m_mean.values, "o-", color=c, lw=2, ms=6, label=lbl)
    ax_mon.set_title(f"월별 평균 {target_col}", fontsize=11, fontweight="bold")
    ax_mon.set_xlabel("월"); ax_mon.set_ylabel(ylabel)
    ax_mon.set_xticks(range(1, 13))
    ax_mon.legend(fontsize=8)
    ax_mon.grid(True, alpha=0.3, axis="y")

    # (E) 강수량 vs 지표 산점도
    ax_rain = fig.add_subplot(gs[2, 1])
    for lbl, s, c in [("Train", s_train, COLORS["train"]),
                      ("Val",   s_val,   COLORS["val"]),
                      ("Test",  s_test,  COLORS["test"])]:
        r = rain_sum.reindex(s.index).fillna(0)
        mask = r > 0
        if mask.sum() > 0:
            ax_rain.scatter(r[mask], s[mask], color=c, alpha=0.3, s=8,
                            label=f"{lbl} (강수시)")
    ax_rain.set_title(f"강수량 vs {target_col} (강수 발생 시점만)",
                      fontsize=11, fontweight="bold")
    ax_rain.set_xlabel("강수량 합계 (mm/30min, 3관측소)")
    ax_rain.set_ylabel(ylabel)
    ax_rain.legend(fontsize=8)
    ax_rain.grid(True, alpha=0.3)

    fig.suptitle(f"{target_col} 구간별 분포 분석 ({ratio_str})",
                 fontsize=14, fontweight="bold")

    fname = f"{target_col.lower().replace('_vu', '')}_split_distribution.png"
    plt.savefig(str(RESULTS_DIR / fname), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"저장: {RESULTS_DIR / fname}")

In [ ]:
# ── 데이터 로드 (최초 1회 실행, 이후 셀에서 tms_30 재사용) ──────
tms_30 = load_tms_resampled()
print("사용 가능한 TMS 컬럼:", [c for c in tms_30.columns if "_VU" in c])

In [ ]:
# ── 단일 지표 분석: Cell 1의 TARGET 변수로 선택 ─────────────────
analyze_target(TARGET, tms_30)

In [ ]:
# ── 전체 TMS 지표 순차 분석 ─────────────────────────────────────
TMS_TARGETS = ["TOC_VU", "SS_VU", "TN_VU", "TP_VU", "FLUX_VU", "PH_VU"]

for target in TMS_TARGETS:
    analyze_target(target, tms_30)